# Bab 3: Python untuk Data Mining

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab03/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab03")
print("Folder kerja:", os.getcwd())

## Listing 3.1: Operasi dasar array NumPy

`kode/bab03/numpy_dasar.py`

In [ ]:
import numpy as np

harga = np.array([22000, 28000, 20000, 18000, 32000])
qty = np.array([3, 1, 2, 4, 1])

omzet = harga * qty
print("omzet per menu :", omzet)
print("total omzet    :", omzet.sum())

diskon = np.array([0.0, 0.1, 0.2])
harga_promo = harga[:, None] * (1 - diskon[None, :])
print("bentuk tabel   :", harga_promo.shape)
print(harga_promo[:2])

mahal = harga >= 25000
print("menu mahal     :", harga[mahal])
print("rata-rata, simpangan baku:",
      harga.mean(), round(harga.std(), 1))

## Listing 3.2: Membaca dan memeriksa data transaksi

`kode/bab03/baca_data.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
kasir = pd.read_csv(DATA + "transaksi_kasir_2026q1.csv")
app = pd.read_csv(DATA + "pesanan_aplikasi_2026q1.csv")

print(kasir.shape, app.shape)
print(kasir.iloc[:3, :5].to_string())
print()
kasir.info()
print()
print(kasir["metode_bayar"].value_counts())

## Listing 3.3: Menyeragamkan skema dan menyatukan dua sumber

`kode/bab03/seragamkan.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
kasir = pd.read_csv(DATA + "transaksi_kasir_2026q1.csv",
                    parse_dates=["waktu"])
app = pd.read_csv(DATA + "pesanan_aplikasi_2026q1.csv")
gerai = pd.read_csv(DATA + "gerai.csv")
menu = pd.read_csv(DATA + "menu.csv")

app = app.rename(columns={
    "order_id": "no_struk", "created_at": "waktu",
    "quantity": "qty", "unit_price": "harga_satuan",
    "promo_amount": "diskon", "payment": "metode_bayar",
    "member_id": "id_anggota"})
app["waktu"] = (pd.to_datetime(app["waktu"])
                .dt.tz_localize(None))
peta_gerai = dict(zip(gerai["kode_aplikasi"], gerai["id_gerai"]))
peta_menu = dict(zip(menu["nama_menu"], menu["kode_menu"]))
app["kode_gerai"] = app["outlet"].map(peta_gerai)
app["kode_menu"] = app["item_name"].map(peta_menu)
app["metode_bayar"] = app["metode_bayar"].replace({
    "e-wallet": "Dompet Digital", "qris": "QRIS",
    "card": "Kartu Kredit", "virtual_account": "Transfer VA"})

kolom = ["no_struk", "waktu", "kode_gerai", "kode_menu", "qty",
         "harga_satuan", "diskon", "metode_bayar", "id_anggota"]
transaksi = pd.concat(
    [kasir[kolom].assign(channel="gerai", sumber="kasir"),
     app[kolom + ["channel"]].assign(sumber="aplikasi")],
    ignore_index=True)

print(transaksi.shape)
print(transaksi.isna().sum().to_string())
print(transaksi["sumber"].value_counts().to_string())
transaksi.to_csv(DATA + "transaksi_gabungan_2026q1.csv",
                 index=False)

## Listing 3.4: Menggabungkan transaksi dengan tabel gerai dan menu

`kode/bab03/gabung_dimensi.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
trx = pd.read_csv(DATA + "transaksi_gabungan_2026q1.csv",
                  parse_dates=["waktu"])
gerai = pd.read_csv(DATA + "gerai.csv")
menu = pd.read_csv(DATA + "menu.csv")

trx = trx.merge(
    gerai[["id_gerai", "kota", "tipe_lokasi"]],
    left_on="kode_gerai", right_on="id_gerai",
    how="left", validate="many_to_one", indicator=True)
print(trx["_merge"].value_counts().to_string())
trx = trx.drop(columns=["id_gerai", "_merge"])
trx = trx.merge(menu[["kode_menu", "nama_menu", "kategori"]],
                on="kode_menu", how="left",
                validate="many_to_one")

trx["nilai"] = trx["qty"] * trx["harga_satuan"] - trx["diskon"]
print(trx[["no_struk", "kota", "nama_menu", "qty",
           "nilai"]].head(4).to_string())

per_struk = (trx.groupby("no_struk")
             .agg(item=("qty", "sum"), nilai=("nilai", "sum")))
print(per_struk.describe().round(0).to_string())
trx.to_csv(DATA + "transaksi_lengkap_2026q1.csv", index=False)

## Listing 3.5: Operasi OLAP dengan pandas

`kode/bab03/olap.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
trx = pd.read_csv(DATA + "transaksi_lengkap_2026q1.csv",
                  parse_dates=["waktu"])
trx["bulan"] = trx["waktu"].dt.strftime("%Y-%m")
trx["jam"] = trx["waktu"].dt.hour

kubus = trx.pivot_table(index="kota", columns="bulan",
                        values="nilai", aggfunc="sum")
print((kubus / 1e6).round(1).to_string())

print((kubus.sum() / 1e6).round(1).to_string())

pku = trx[trx["kota"] == "Pekanbaru"]
print((pku.pivot_table(index="kode_gerai", columns="bulan",
                       values="nilai", aggfunc="sum")
       / 1e6).round(1).to_string())

ramadan = trx["waktu"].between("2026-02-18", "2026-03-19 23:59")
irisan = trx[trx["bulan"] == "2026-01"]
print(irisan.groupby("kategori")["nilai"].sum()
      .div(1e6).round(1).to_string())
dadu = trx[ramadan & (trx["jam"] >= 17)
           & trx["kota"].isin(["Pekanbaru", "Padang"])]
print(dadu.pivot_table(index="kota", columns="kategori",
                       values="nilai", aggfunc="sum")
      .div(1e6).round(1).to_string())

## Listing 3.6: Grafik omzet harian dan peta panas jam–hari

`kode/bab03/grafik.py`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA = "kode/data/"
trx = pd.read_csv(DATA + "transaksi_lengkap_2026q1.csv",
                  parse_dates=["waktu"])
harian = trx.set_index("waktu")["nilai"].resample("D").sum()

fig, ax = plt.subplots(figsize=(8, 3))
(harian / 1e6).plot(ax=ax)
ax.axvspan("2026-02-18", "2026-03-19", alpha=0.15)
ax.set_ylabel("omzet (juta Rp)")
fig.savefig("omzet_harian.png", dpi=150, bbox_inches="tight")

trx["hari"] = trx["waktu"].dt.dayofweek
trx["jam"] = trx["waktu"].dt.hour
biasa = trx[trx["waktu"] < "2026-02-18"]
peta = biasa.pivot_table(index="hari", columns="jam",
                         values="no_struk", aggfunc="nunique")
fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(peta, cmap="YlOrBr", ax=ax)
fig.savefig("peta_jam_hari.png", dpi=150, bbox_inches="tight")
print((harian / 1e6).describe().round(2).to_string())